# 04 — NER — Gemini free tier few-shot

Gán nhãn **PER, LOC, ORG, TITLE, TME, NUM** (+ **DYNASTY**). Gọi theo lô để tiết kiệm quota.
Lấy API key free: https://aistudio.google.com/app/apikey

Vào: `outputs/<docid>/<docid>_seg.tsv` → Ra: `outputs/<docid>/<docid>_ner.json`.

In [ ]:
# %pip install google-generativeai

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))   # để import module trong src/

DATA_DIR    = PROJECT_ROOT / 'data'
RAW_IMG_DIR = DATA_DIR / 'raw_images' / 'downloaded_images'   # ảnh gốc theo <docid>/
PREP_DIR    = DATA_DIR / 'preprocessed'
OCR_DIR     = DATA_DIR / 'ocr'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
for d in [PREP_DIR, OCR_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def list_works(base):
    """Danh sách 'work id' = tên thư mục con trong base (mỗi tác phẩm/quyển = 1 folder)."""
    if not base.exists():
        return []
    return sorted(p.name for p in base.iterdir() if p.is_dir())

def images_of(base, wid):
    d = base / wid
    return sorted(list(d.glob('*.jpg')) + list(d.glob('*.png')))

print('Project root:', PROJECT_ROOT)
print('Ảnh gốc     :', RAW_IMG_DIR)

In [ ]:
import os, json, time, re
import google.generativeai as genai
from tqdm.auto import tqdm

genai.configure(api_key=os.environ.get('GOOGLE_API_KEY') or 'DÁN_API_KEY_VÀO_ĐÂY')
ner_model = genai.GenerativeModel('gemini-2.5-flash')
NER_DELAY, BATCH = 4.5, 10
LABELS = ['PER', 'LOC', 'ORG', 'TITLE', 'TME', 'NUM', 'DYNASTY']
NER_PROMPT = (
    'Bạn là chuyên gia trích xuất thực thể trong văn bản Hán cổ về LỊCH SỬ VIỆT NAM. '
    'Bộ nhãn: PER=nhân danh, LOC=địa danh, ORG=tổ chức, TITLE=chức tước/tước hiệu, '
    'TME=thời gian (niên hiệu, ngày tháng, mùa), NUM=số lượng, DYNASTY=triều đại. '
    'Quy tắc: text phải xuất hiện NGUYÊN VĂN trong câu; không suy diễn; nếu không có thì mảng rỗng. '
    'TRẢ VỀ DUY NHẤT JSON array, phần tử {"id": <id>, "entities": [{"text":"..","label":".."}]}. '
    'Không markdown, không giải thích.\n\nDữ liệu:\n'
)

def read_seg(wid):
    p = OUTPUTS_DIR / wid / f'{wid}_seg.tsv'; rows = []
    if p.exists():
        for ln in p.read_text(encoding='utf-8').splitlines():
            if '\t' in ln:
                sid, sent = ln.split('\t', 1); rows.append((sid, sent))
    return rows

In [ ]:
def parse_json(txt):
    txt = re.sub(r'^```(json)?|```$', '', txt.strip(), flags=re.MULTILINE).strip()
    m = re.search(r'\[.*\]', txt, re.DOTALL)
    try:
        return json.loads(m.group(0) if m else txt)
    except Exception:
        return []

def ner_batch(batch):
    payload = [{'id': sid, 'sentence': s} for sid, s in batch]
    prompt = NER_PROMPT + json.dumps(payload, ensure_ascii=False)
    for _ in range(3):
        try:
            r = ner_model.generate_content(prompt)
            data = parse_json(r.text or '')
            return {d.get('id'): d.get('entities', []) for d in data if isinstance(d, dict)}
        except Exception as e:
            print('  ! NER lỗi, thử lại 20s:', str(e)[:90]); time.sleep(20)
    return {}

def validate(sentence, ents):
    out = []
    for e in ents or []:
        t, l = e.get('text', ''), e.get('label', '')
        if t and l in LABELS and t in sentence:
            out.append({'text': t, 'label': l})
    return out

### Chạy NER → ghi `_ner.json`

In [ ]:
works = list_works(OUTPUTS_DIR)
for wid in works:
    rows = read_seg(wid)
    if not rows:
        continue
    sent_map = dict(rows); results = {}
    for i in tqdm(range(0, len(rows), BATCH), desc=wid):
        batch = rows[i:i+BATCH]
        ent_map = ner_batch(batch)
        for sid, _ in batch:
            results[sid] = validate(sent_map[sid], ent_map.get(sid, []))
        time.sleep(NER_DELAY)
    ner_json = [{'sentence_id': sid, 'sentence': sent_map[sid], 'entities': results.get(sid, [])} for sid, _ in rows]
    out = OUTPUTS_DIR / wid / f'{wid}_ner.json'
    out.write_text(json.dumps(ner_json, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'{wid}: {len(ner_json)} câu, {sum(len(x["entities"]) for x in ner_json)} thực thể -> {out}')

In [ ]:
works = list_works(OUTPUTS_DIR)
if works:
    p = OUTPUTS_DIR / works[0] / f'{works[0]}_ner.json'
    if p.exists():
        for item in json.loads(p.read_text(encoding='utf-8'))[:5]:
            print(json.dumps(item, ensure_ascii=False, indent=2))